# 1.6 Grounding the Request in Graph Entities: Does Linking Question Entities to Graph IDs Improve Answers?

*Formerly notebook 21.*

> **Outputs in this copy come from the confidence run** (`SMASH_MODE=batch`): 1,000 2Wiki and 3 × 300 MetaQA
> questions; new calls $0.33 via the Batch API (it reuses notebook 1.4a's and 19's caches). Task 1.6. They are reported
> in `RESULTS-REPORT.md` and the v2 paper.

**Presentation map:** Task 1.6 (grounding the request in graph entities), Slides 40-41 (traversal starts from the
topic entity), Notebooks A1, 1.4a and 1.2b (the graphs and retrievers reused here).

Every graph method in the earlier notebooks starts from an entity of the graph: the MetaQA agent from the entity in
`[brackets]`, HippoRAG-style retrieval from the nodes most similar to the question. In practice nobody marks the
entity for the agent: the names in a request have to be **linked** to graph identifiers first. This notebook measures
how often that linking fails and what it costs downstream.

## Protocol (frozen before the test run)

**Research question.** Does linking the entities of a question to graph IDs before retrieval or reasoning improve the
answers, and how often does linking fail?

**Hypotheses.** (H1) A cascade over the graph's entity names links the question entity correctly more often than the
LLM naming the entity on its own. (H2) Answers with predicted links are better than answers from the raw question
text and close to answers with gold links; the gap gold − predicted is the downstream cost of linking errors.

**Data** (local, reused):
- **MetaQA** test (vanilla), **300 questions per hop** (1/2/3-hop), sampled with seed 7 from the full test splits by
  Notebook A1's procedure (all topic types: films, people, tags, years). The topic entity is marked in `[brackets]`:
  **the brackets are removed before linking and before answering**; the bracketed name is kept only as the gold link.
  Graph: the curated WikiMovies KB (43,234 entities, Notebooks A1/1.2b).
- **2WikiMultiHopQA** dev, the **1,000 questions** of Notebook 1.4a, with Notebook 1.4a's passage pool, embeddings and
  LLM-extracted graph (read from its cache, nothing rebuilt). Gold links: the subjects of the `evidences` triples that
  are not the object of another evidence triple (the entities the question names; 1 for compositional/inference
  questions, 2 for comparison questions), mapped to the graph's entity node with the same normalised name.

**Linking conditions.**
- **Cascade (graph method):** exact → alias → embedding → LLM, over the graph's entity names.
  1. *exact*: maximal n-grams of the normalised question equal to a normalised entity name; n-grams that occur in
     ≥ 1 % of the benchmark's (unlabelled) questions are template words and are skipped.
     Exactly one candidate → linked.
  2. *alias* (only if no exact candidate): the same with alias keys (articles and "and" dropped, parentheticals
     dropped). Exactly one candidate → linked.
  3. *embedding* (only if no exact or alias candidate): every non-template n-gram (≤ 4 tokens) is embedded
     (`text-embedding-3-small`) and compared with the embedded entity names; exactly one entity with cosine ≥ 0.9
     (Notebook 1.4a's synonym threshold) → linked.
  4. *LLM disambiguation* (all other cases: 0 or ≥ 2 candidates): the LLM picks from the exact/alias candidates plus
     the 10 entities closest by embedding, each shown with a one-line graph description (MetaQA: one topic entity;
     2Wiki: every entity the question names), or none.
- **LLM-named (baseline):** the LLM names the entity (MetaQA) or entities (2Wiki) the question is about, without seeing
  the graph; the names are matched to the graph by normalised name only.
- **Gold:** the bracketed MetaQA topic; the 2Wiki evidence subjects.

**Downstream conditions (same reader, only the grounding differs).**
- **MetaQA:** Notebook 1.2b's graph agent over the curated KB (tool `get_neighbors`, 8 tool steps, then a final answer),
  given the bracket-free question and, except in *raw*, a line `Topic entity: <name>`: **raw text** (no hint; the
  agent must find the entity itself), **gold link**, **predicted link** (cascade; no hint if the cascade links
  nothing), **LLM-named** (the LLM's name, linked or not).
- **2Wiki:** Notebook 1.4a's reader (top-5 passages, same prompt). Raw-text retrieval: **vector** (dense top 5) and
  **graph-raw** (Notebook 1.4a's PPR seeded from the question through its most similar triples, HippoRAG 2's query→triple
  linking). Linked retrieval: the same PPR **seeded at the linked entity nodes** (weight 1 each, plus Notebook 1.4a's
  small dense passage prior), with gold, predicted (cascade) and LLM-named links. The raw-text answers are Notebook 1.4a's
  cached answers (identical requests).

**Metrics.** *Linking:* linking accuracy (MetaQA: predicted topic = gold; 2Wiki: predicted entity set = gold set, on
questions whose gold entities are all in the graph), with Wilson intervals; entity-level precision/recall (2Wiki);
share of questions per cascade stage; failure types (no link, wrong link). *Downstream:* **Hits@1** (MetaQA) and
**token F1** (2Wiki); secondary: MetaQA F1, 2Wiki EM and passage recall@5, tokens and tool calls.

**Primary metric and analysis.** Primary: the downstream difference **predicted link − raw text** (MetaQA Hits@1,
pooled over hops; 2Wiki F1 against graph-raw), paired 95 % bootstrap over questions. Linking accuracy (cascade vs
LLM-named, paired) is the primary measure of the second half of the question. Reported next to them: gold − predicted,
LLM-named − raw, predicted − vector (2Wiki), per hop / question type.

**n.** 900 MetaQA + 1,000 2Wiki questions. Model `gpt-6-luna`, no reasoning, temperature 0, Batch API.

**Threats.** MetaQA marks the topic in brackets and its questions are templates; the brackets are removed, but the
template-word filter uses question statistics (no labels). MetaQA gold names are exact KB names, so exact match is
favoured; 2Wiki's graph is LLM-extracted, so a gold entity may be missing or split. The cascade's thresholds (1 %,
0.9) are fixed a priori, not tuned. The raw-text MetaQA agent can still find the entity by guessing names through
its tool, so *raw* is a strong baseline. One model, one prompt per stage, one run.

**Protocol changes after the test run.**
1. 2Wiki gold links: the evidence names are Wikidata labels and in 2 of 25 test questions differed from the name used
   in the question and the article title ("Maurice of Nassau" / "Maurice, Prince of Orange"; "Lisbet" / "Lisbeth
   Palme"), and one graph node carried the year as a parenthetical. Each gold entity now also accepts the title of a
   supporting paragraph that matches no other evidence entity, and names are also compared without a trailing
   parenthetical. The rule applies to the cascade and the LLM-named baseline alike; nothing else changed.

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, string, time, random, hashlib, unicodedata
from collections import Counter, defaultdict
import numpy as np
import scipy.sparse as sp
import networkx as nx
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot, PRICE, EMBED_PRICE

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
MQ, WK = "data/benchmarks/metaqa", "data/benchmarks/musique_2wiki"
llm_m = LLM(f"{MQ}/cache_21" + ("_test" if TEST else ""))        # this notebook's MetaQA stages
llm_w = LLM(f"{WK}/cache" + ("_test" if TEST else ""))           # Notebook 1.4a's cache: read its stages, write only nb21_*
SEED, N_PER_HOP, N_TEST = 7, 300, 25
N_W = N_TEST if TEST else 1000
print("MODE =", MODE)

MODE = batch


## MetaQA: the curated KB and the questions

The KB and the question loader are Notebook 1.2b's; the sample is Notebook A1's procedure at 300 per hop (all topic
types). Test mode keeps the first 25 of each hop's sample, so the test questions are a subset of the full run's.

In [2]:
RELS = ["directed_by", "written_by", "starred_actors", "release_year", "in_language", "has_genre", "has_tags",
        "has_imdb_rating", "has_imdb_votes"]

def key(s):
    # Notebook 1.2b's linking and scoring key: lower case, accents and punctuation removed, whitespace collapsed.
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", s)).strip()

KB = nx.MultiDiGraph()
for line in open(f"{MQ}/kb.txt", encoding="utf-8"):
    h, r, t = line.rstrip("\n").split("|")
    KB.add_edge(h, t, key=r)
movies_kb = {h for h, _, _ in KB.edges(keys=True)}

def load_split(hop):
    rows = []
    for i, line in enumerate(open(f"{MQ}/qa_test_{hop}hop.txt", encoding="utf-8")):
        q, a = line.rstrip("\n").split("\t")
        rows.append({"qid": f"{hop}hop-{i}", "hop": hop, "question": q, "text": q.replace("[", "").replace("]", ""),
                     "topic": re.search(r"\[(.+?)\]", q).group(1), "gold": a.split("|")})
    return rows

splits = {h: load_split(h) for h in (1, 2, 3)}
rng = random.Random(SEED)
sample = {h: rng.sample(splits[h], N_PER_HOP) for h in (1, 2, 3)}
QM = [q for h in (1, 2, 3) for q in (sample[h][:N_TEST] if TEST else sample[h])]

def topic_type(e):
    if e in movies_kb:
        return "film"
    rels = {r for _, _, r in KB.in_edges(e, keys=True)}
    return "person" if rels & {"directed_by", "written_by", "starred_actors"} else "tag/other"
for q in QM:
    q["ttype"] = topic_type(q["topic"])
print(f"MetaQA KB: {KB.number_of_nodes():,} entities, {KB.number_of_edges():,} triples")
print(f"{len(QM)} questions (" + ", ".join(f"{h}-hop {sum(q['hop'] == h for q in QM)}" for h in (1, 2, 3)) +
      "); topic types: " + str(dict(Counter(q["ttype"] for q in QM))))
print("Example:", QM[0]["question"], "->", repr(QM[0]["text"]), "| gold link:", QM[0]["topic"])

MetaQA KB: 43,234 entities, 133,582 triples
900 questions (1-hop 300, 2-hop 300, 3-hop 300); topic types: {'film': 566, 'person': 318, 'tag/other': 16}
Example: what are some words that describe movie [Blue Like Jazz] -> 'what are some words that describe movie Blue Like Jazz' | gold link: Blue Like Jazz


## 2WikiMultiHopQA: Notebook 1.4a's passage pool, embeddings and graph (read from its cache)

The next cells are Notebook 1.4a's code for 2Wiki, unchanged (as in Notebook 1.5), so that every embedding and extraction
request matches its cache and nothing is recomputed. `build_graph` additionally keeps the entity-name embeddings
(`d["E"]`), which the cascade reuses.

In [3]:
NAME = "2wikimultihopqa"
W = {"name": NAME}
qs_all = json.load(open(f"{WK}/{NAME}.json"))
corpus = json.load(open(f"{WK}/{NAME}_corpus.json"))
for q in qs_all:
    q["qid"], q["golds"] = q["_id"], [q["answer"]]
    ctx_ = {t: " ".join(s) for t, s in q["context"]}
    q["support"] = {t + "\n" + ctx_[t] for t in {t for t, _ in q["supporting_facts"]} if t in ctx_}
    q["group"] = q["type"]
W["questions"] = qs_all[:N_W]
passages = [p["title"] + "\n" + p["text"] for p in corpus]
if TEST:
    keep = set()
    for q in W["questions"]:
        keep |= {t + "\n" + " ".join(s) for t, s in q["context"]}
    passages = [p for p in passages if p in keep]
W["passages"], W["pid"] = passages, {p: i for i, p in enumerate(passages)}
W["P"] = llm_w.embed(f"{NAME}_passages", W["passages"])
W["Q"] = llm_w.embed(f"{NAME}_questions", [q["question"] for q in W["questions"]])
print(f"2Wiki: {len(W['questions']):,} questions, {len(passages):,} passages in the pool, "
      f"question types {dict(Counter(q['group'] for q in W['questions']))}")

2Wiki: 1,000 questions, 6,119 passages in the pool, question types {'compositional': 413, 'comparison': 244, 'bridge_comparison': 235, 'inference': 108}


In [4]:
class Triple(BaseModel):
    subject: str
    predicate: str
    object: str

class OpenIE(BaseModel):
    entities: list[str] = Field(description="named entities, dates and numbers mentioned in the passage")
    triples: list[Triple] = Field(description="factual subject-predicate-object triples stated in the passage")

OPENIE_SYS = ("Extract an open knowledge graph from a Wikipedia passage. List the named entities (people, places, "
              "organisations, works, events, dates, numbers) and every factual triple the passage states. Use full, "
              "canonical entity names (resolve pronouns to the entity they refer to); keep dates and numbers exact. At most "
              "40 triples, most important first.")

TRIPLE_RE = re.compile(r'\{\s*"subject":\s*"((?:[^"\\]|\\.)*)",\s*"predicate":\s*"((?:[^"\\]|\\.)*)",\s*"object":\s*"((?:[^"\\]|\\.)*)"\s*\}')

def salvage(text):
    # An output cut off at the token limit still contains complete triples: keep those.
    return {"entities": [], "triples": [{"subject": a, "predicate": b, "object": c} for a, b, c in TRIPLE_RE.findall(text)]}

reqs = {f"p{i}": LLM.body([{"role": "system", "content": OPENIE_SYS}, {"role": "user", "content": p}],
                          max_tokens=4000, schema=OpenIE)
        for i, p in enumerate(W["passages"])}
res = llm_w.run(f"{NAME}_openie", reqs, parse=OpenIE)
W["openie"] = [res[f"p{i}"]["parsed"] or salvage(res[f"p{i}"].get("text", "")) for i in range(len(W["passages"]))]

norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(s).lower())).strip()

def build_graph(d):
    # Nodes: passages 0..P-1, then entities. Edges: entity-entity (triple), entity-passage (mention), synonyms.
    n_p = len(d["passages"])
    ent_id, ent_name = {}, []
    def eid(s):
        k = norm(s)
        if not k:
            return None
        if k not in ent_id:
            ent_id[k] = n_p + len(ent_name); ent_name.append(s)
        return ent_id[k]
    rows, cols, facts = [], [], []
    for pi, ex in enumerate(d["openie"]):
        for e in ex["entities"]:
            j = eid(e)
            if j is not None:
                rows += [pi, j]; cols += [j, pi]
        for t in ex["triples"]:
            a, b = eid(t["subject"]), eid(t["object"])
            if a is None or b is None or a == b:
                continue
            rows += [a, b, pi, a, pi, b]; cols += [b, a, a, pi, b, pi]
            facts.append((a, b, pi, f"{t['subject']} {t['predicate']} {t['object']}"))
    d["n_p"], d["ent_name"], d["facts"] = n_p, ent_name, facts
    # Synonym edges between near-duplicate entity names (HippoRAG 2), computed on entity-name embeddings.
    E = llm_w.embed(f"{d['name']}_entities", ent_name, cap=200)
    d["E"] = E                                       # kept for the linking cascade (Notebook 1.6)
    syn = 0
    for s in range(0, len(ent_name), 512):
        S = E[s:s + 512] @ E.T
        ii, jj = np.where(S >= 0.9)
        for a, b in zip(ii + s, jj):
            if a < b:
                rows += [n_p + a, n_p + b]; cols += [n_p + b, n_p + a]; syn += 1
    n = n_p + len(ent_name)
    A = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n, n))
    A.data[:] = 1.0                                  # duplicate edges count once
    deg = np.asarray(A.sum(1)).ravel(); deg[deg == 0] = 1
    d["T"] = (sp.diags(1 / deg) @ A).T.tocsr()        # column-stochastic transition matrix
    d["F"] = llm_w.embed(f"{d['name']}_facts", [f[3] for f in facts], cap=300)
    return n, syn

n, syn = build_graph(W)
print(f"2Wiki graph: {W['n_p']:,} passage + {len(W['ent_name']):,} entity nodes, {len(W['facts']):,} triples, "
      f"{syn:,} synonym edges")

2Wiki graph: 6,119 passage + 68,410 entity nodes, 63,711 triples, 10,448 synonym edges


### Gold links for 2Wiki

The entities a 2Wiki question names are the subjects of its `evidences` triples that are not the object of another
evidence triple (the film or person of a compositional question; both films of a comparison). Each is mapped to the
graph's entity node with the same normalised name (Notebook 1.4a's `norm`, the key under which the graph merges names).
The evidence names are Wikidata labels and sometimes differ from the name in the question and the article title
("Maurice of Nassau" vs. "Maurice, Prince of Orange"; "Lisbet" vs. "Lisbeth Palme"). Each gold entity therefore also
accepts the title of a supporting paragraph that matches no other evidence entity, and every accepted name also
without a trailing parenthetical ("… (1954)").

In [5]:
W["key2ent"] = {norm(nm): j for j, nm in enumerate(W["ent_name"])}
strip_par = lambda s: re.sub(r"\s*\([^)]*\)\s*$", "", s)

def gold_names(q):
    objs = {o for _, _, o in q["evidences"]}
    return list(dict.fromkeys(s for s, _, _ in q["evidences"] if s not in objs))

def accepted_keys(q):
    # Per gold entity: its evidence name, plus supporting-paragraph titles that match no other evidence entity
    # (Wikidata label vs. article title), each also without a trailing parenthetical.
    roots = q["gold_names"]
    named = {norm(x) for e in q["evidences"] for x in (e[0], e[2])}
    left = [t for t in dict.fromkeys(t for t, _ in q["supporting_facts"]) if norm(t) not in named]
    out = []
    for r in roots:
        names = [r] + (left if norm(r) not in {norm(t) for t, _ in q["supporting_facts"]} else [])
        out.append(list(dict.fromkeys(k for nm in names for k in (norm(nm), norm(strip_par(nm))) if k)))
    return out

def ent_key(j):
    return norm(strip_par(W["ent_name"][j]))

for q in W["questions"]:
    q["gold_names"] = gold_names(q)
    q["gold_acc"] = accepted_keys(q)
    q["gold_ents"] = [next((W["key2ent"][k] for k in acc if k in W["key2ent"]), None) for acc in q["gold_acc"]]
    q["gold_in_graph"] = all(j is not None for j in q["gold_ents"])
cov = np.mean([q["gold_in_graph"] for q in W["questions"]])
print(f"gold entities per question: {dict(Counter(len(q['gold_names']) for q in W['questions']))}; "
      f"questions whose gold entities are all graph nodes: {cov:.1%}")
q = W["questions"][0]
print("Example:", q["question"], "-> gold link(s):", q["gold_names"])

gold entities per question: {1: 490, 2: 466, 3: 40, 0: 1, 4: 3}; questions whose gold entities are all graph nodes: 95.6%
Example: When did Lothair Ii's mother die? -> gold link(s): ['Lothair II']


## The linking cascade

One implementation for both graphs. A *catalogue* holds the graph's entity names, their normalised keys and alias keys,
and their name embeddings (MetaQA: embedded here, stage `nb21_metaqa_names`; 2Wiki: Notebook 1.4a's entity-name
embeddings). Template words are n-grams that occur in at least 1 % of all of the benchmark's questions (MetaQA: the
39,093 test questions without brackets; 2Wiki: the 1,000 questions), computed without labels.

In [6]:
STOP = {"a", "an", "the", "and", "of", "in", "on", "for", "to", "by", "with", "is", "was", "are", "were", "what",
        "which", "who", "whom", "whose", "when", "where", "did", "does", "do", "that", "this", "or", "as", "at", "from"}
TEMPLATE_DF, EMB_THRESHOLD, MAX_EMB_N, N_EMB_CANDS = 0.01, 0.9, 4, 10

def alias_key(k):
    return " ".join(t for t in k.split() if t not in {"a", "an", "the", "and"})

class Catalogue:
    def __init__(self, label, names, keyf, E, desc, all_questions):
        self.label, self.names, self.keyf, self.E, self.desc = label, names, keyf, E, desc
        self.key2ids, self.alias2ids = defaultdict(list), defaultdict(list)
        for i, nm in enumerate(names):
            k = keyf(nm)
            if k:
                self.key2ids[k].append(i)
            for a in {alias_key(k), alias_key(keyf(re.sub(r"\s*\([^)]*\)", "", nm)))}:
                if a:
                    self.alias2ids[a].append(i)
        self.maxn = min(15, max(len(k.split()) for k in self.key2ids))
        df = Counter()
        for t in all_questions:
            toks = keyf(t).split()
            df.update({" ".join(toks[i:i + n]) for i in range(len(toks))
                       for n in range(1, min(self.maxn, len(toks) - i) + 1)})
        self.template = {g for g, c in df.items() if c >= TEMPLATE_DF * len(all_questions)}

    def matches(self, text, table):
        toks = self.keyf(text).split()
        m = [(i, i + n, g) for i in range(len(toks)) for n in range(1, min(self.maxn, len(toks) - i) + 1)
             for g in [" ".join(toks[i:i + n])] if g in table and g not in self.template]
        maximal = [a for a in m if not any(b[0] <= a[0] and a[1] <= b[1] and b[1] - b[0] > a[1] - a[0] for b in m)]
        return list(dict.fromkeys(i for _, _, g in maximal for i in table[g]))

    def ngrams(self, text):
        toks = self.keyf(text).split()
        return list(dict.fromkeys(g for i in range(len(toks)) for n in range(1, min(MAX_EMB_N, len(toks) - i) + 1)
                                  for g in [" ".join(toks[i:i + n])]
                                  if g not in self.template and not set(g.split()) <= STOP))

def cascade_first_pass(cat, texts, llm, stage):
    """Stages 1-3 for every question. Returns per question {stage, ids, cands}; stage 'llm' still needs a pick."""
    out = []
    grams = {i: cat.ngrams(t) for i, t in enumerate(texts)}
    uniq = sorted({g for v in grams.values() for g in v})
    G_E = llm.embed(stage, uniq) if uniq else np.zeros((0, cat.E.shape[1]), np.float32)
    gidx = {g: i for i, g in enumerate(uniq)}
    for i, t in enumerate(texts):
        ex = cat.matches(t, cat.key2ids)
        al = cat.matches(t, cat.alias2ids) if not ex else []
        if grams[i]:
            sims = (G_E[[gidx[g] for g in grams[i]]] @ cat.E.T).max(0)
            top = np.argsort(-sims)[:N_EMB_CANDS]
            emb = [(int(j), float(sims[j])) for j in top]
        else:
            emb = []
        if len(ex) == 1:
            r = {"stage": "exact", "ids": ex}
        elif not ex and len(al) == 1:
            r = {"stage": "alias", "ids": al}
        elif not ex and not al and sum(s >= EMB_THRESHOLD for _, s in emb) == 1:
            r = {"stage": "embedding", "ids": [j for j, s in emb if s >= EMB_THRESHOLD]}
        else:
            r = {"stage": "llm", "ids": []}
        r["cands"] = list(dict.fromkeys((ex or al)[:N_EMB_CANDS] + [j for j, _ in emb]))[:N_EMB_CANDS + 5]
        r["n_exact"], r["n_alias"] = len(ex), len(al)
        out.append(r)
    return out

### Catalogues: one-line descriptions for the LLM step

MetaQA: Notebook A1's entity document (its relations and neighbours), cut to 160 characters. 2Wiki: the name and up
to two extracted facts that mention it.

In [7]:
PHRASE = {"directed_by": ("directed by", "director of"), "written_by": ("written by", "writer of"),
          "starred_actors": ("starring", "acted in"), "release_year": ("released in", "release year of"),
          "in_language": ("in language", "language of"), "has_genre": ("genre", "genre of"),
          "has_tags": ("tagged", "tag of"), "has_imdb_votes": ("IMDb votes", "IMDb votes of"),
          "has_imdb_rating": ("IMDb rating", "IMDb rating of")}

def entity_doc(e, max_chars=160):
    # Notebook A1's entity document, shorter.
    groups = defaultdict(list)
    for _, t, r in KB.out_edges(e, keys=True):
        groups[PHRASE[r][0]].append(t)
    for s, _, r in KB.in_edges(e, keys=True):
        groups[PHRASE[r][1]].append(s)
    body = "; ".join(f"{k}: {', '.join(v)}" for k, v in groups.items())
    return f"{e} — {body}"[:max_chars]

MQ_NAMES = sorted(KB.nodes())
CAT_M = Catalogue("MetaQA", MQ_NAMES, key, llm_m.embed("nb21_metaqa_names", MQ_NAMES, cap=200),
                  lambda i: entity_doc(MQ_NAMES[i]), [q["text"] for h in (1, 2, 3) for q in splits[h]])

ent_facts = defaultdict(list)
for a, b, pi, txt in W["facts"]:
    for node in (a, b):
        if len(ent_facts[node - W["n_p"]]) < 2:
            ent_facts[node - W["n_p"]].append(txt)
CAT_W = Catalogue("2Wiki", W["ent_name"], norm, W["E"],
                  lambda j: (W["ent_name"][j] + (" — " + "; ".join(ent_facts[j]) if ent_facts[j] else ""))[:160],
                  [q["question"] for q in qs_all])
for c in (CAT_M, CAT_W):
    print(f"{c.label}: {len(c.names):,} entity names, {len(c.key2ids):,} keys, {len(c.alias2ids):,} alias keys, "
          f"{len(c.template):,} template n-grams, e.g. {sorted(c.template, key=len)[:12]}")

MetaQA: 43,234 entity names, 40,120 keys, 39,826 alias keys, 618 template n-grams, e.g. ['a', 's', 'do', 'on', 'to', 'co', 'of', 'by', 'is', 'as', 'in', 'can']
2Wiki: 68,410 entity names, 68,410 keys, 67,270 alias keys, 590 template n-grams, e.g. ['a', 's', 'i', 'la', 'it', 'do', 'at', 'on', 'to', 'or', 'ii', 'de']


In [8]:
t0 = time.time()
LINK = {"metaqa": cascade_first_pass(CAT_M, [q["text"] for q in QM], llm_m, "nb21_metaqa_ngrams"),
        "2wiki": cascade_first_pass(CAT_W, [q["question"] for q in W["questions"]], llm_w, "nb21_2wiki_ngrams")}
for b, L in LINK.items():
    print(f"{b}: stage reached " + str(dict(Counter(r["stage"] for r in L))) + f"  ({time.time() - t0:.0f} s)")

metaqa: stage reached {'exact': 798, 'llm': 102}  (23 s)
2wiki: stage reached {'exact': 434, 'llm': 564, 'alias': 2}  (23 s)


### Step 4: LLM disambiguation, and the LLM-named baseline

Both are one structured call per question (one Batch job per benchmark and stage). The disambiguation call sees the
bracket-free question and the numbered candidates; the naming call sees only the question.

In [9]:
class Pick(BaseModel):
    chosen: list[int] = Field(description="numbers of the chosen candidates; empty if none fits")

class Named(BaseModel):
    entities: list[str]

PICK_SYS = {
    "metaqa": ("You link a question about movies to a knowledge graph. From the numbered candidate graph entities, "
               "pick the one entity the question starts from (the film, person, tag, year or other entity it names). "
               "Return its number, or an empty list if no candidate is that entity."),
    "2wiki": ("You link a question to a knowledge graph built from Wikipedia. From the numbered candidate graph "
              "entities, pick every entity that the question names explicitly (the entities whose facts must be looked "
              "up to answer it). Return their numbers, or an empty list if no candidate fits.")}
NAME_SYS = {
    "metaqa": ("Name the entity that this question about movies starts from (a film, person, tag, year or other "
               "entity). Return it exactly as it is written in the question, as a list with one name."),
    "2wiki": ("List every entity (person, film, place, work, organisation, ...) that this question names explicitly, "
              "exactly as written in the question.")}
BENCH = {"metaqa": (CAT_M, llm_m, [q["text"] for q in QM], [q["qid"] for q in QM]),
         "2wiki": (CAT_W, llm_w, [q["question"] for q in W["questions"]], [q["qid"] for q in W["questions"]])}

PRED, NAMED = {}, {}
for b, (cat, llm, texts, qids) in BENCH.items():
    L = LINK[b]
    reqs = {}
    for i, r in enumerate(L):
        if r["stage"] == "llm" and r["cands"]:
            cl = "\n".join(f"{n + 1}. {cat.desc(j)}" for n, j in enumerate(r["cands"]))
            reqs[qids[i]] = LLM.body([{"role": "system", "content": PICK_SYS[b]},
                                      {"role": "user", "content": f"Question: {texts[i]}\n\nCandidates:\n{cl}"}],
                                     max_tokens=60, schema=Pick)
    res = llm.run(f"nb21_{b}_disambiguate", reqs, parse=Pick)
    for i, r in enumerate(L):
        if r["stage"] == "llm":
            p = (res.get(qids[i]) or {}).get("parsed") or {"chosen": []}
            ids = [r["cands"][n - 1] for n in p["chosen"] if 1 <= n <= len(r["cands"])]
            r["ids"] = list(dict.fromkeys(ids[:1] if b == "metaqa" else ids))
    PRED[b] = [r["ids"] for r in L]
    named = llm.run(f"nb21_{b}_llm_named", {qids[i]: LLM.body([{"role": "system", "content": NAME_SYS[b]},
                                                               {"role": "user", "content": texts[i]}],
                                                              max_tokens=80, schema=Named)
                                            for i in range(len(texts))}, parse=Named)
    NAMED[b] = [((named[qid].get("parsed") or {}).get("entities") or [])[:1 if b == "metaqa" else 6] for qid in qids]
    print(f"{b}: {len(reqs):,} disambiguation calls ({len(reqs) / len(texts):.0%} of questions), "
          f"{len(texts):,} naming calls")

metaqa: 102 disambiguation calls (11% of questions), 900 naming calls


2wiki: 564 disambiguation calls (56% of questions), 1,000 naming calls


## Linking accuracy

MetaQA: a link is correct when its key equals the gold topic's key (the graph tool resolves names by this key, so
two KB nodes with the same key are one entity for the agent). 2Wiki: the predicted entity set must equal the gold set;
entity-level precision and recall are also shown. The LLM-named baseline is matched to the graph by key only.

In [10]:
def named_to_ids(b, names):
    cat = CAT_M if b == "metaqa" else CAT_W
    return list(dict.fromkeys(i for nm in names for i in cat.key2ids.get(cat.keyf(nm), [])[:1]))

LK = {"metaqa": [], "2wiki": []}
for i, q in enumerate(QM):
    gk = key(q["topic"])
    pk = key(MQ_NAMES[PRED["metaqa"][i][0]]) if PRED["metaqa"][i] else None
    nk = key(NAMED["metaqa"][i][0]) if NAMED["metaqa"][i] else None
    LK["metaqa"].append({"hop": q["hop"], "ttype": q["ttype"], "stage": LINK["metaqa"][i]["stage"],
                         "cascade": float(pk == gk), "cascade_none": float(pk is None),
                         "named": float(nk == gk), "named_in_graph": float(nk in CAT_M.key2ids),
                         "gold_in_cands": float(any(key(MQ_NAMES[j]) == gk for j in LINK["metaqa"][i]["cands"]) or pk == gk)})
def link_eval(q, ids):
    # A predicted entity is correct if its name (or the name without a trailing parenthetical) is an accepted key
    # of a gold entity; the link set is correct if every gold entity is matched and nothing else is predicted.
    keys = [{norm(W["ent_name"][j]), ent_key(j)} for j in ids]
    hit = [any(k & set(acc) for acc in q["gold_acc"]) for k in keys]
    matched = [any(k & set(acc) for k in keys) for acc in q["gold_acc"]]
    return (float(all(matched) and all(hit)), sum(hit) / len(ids) if ids else float("nan"),
            sum(matched) / len(matched) if matched else float("nan"))

for i, q in enumerate(W["questions"]):
    p, nn = PRED["2wiki"][i], named_to_ids("2wiki", NAMED["2wiki"][i])
    row = {"group": q["group"], "stage": LINK["2wiki"][i]["stage"], "gold_in_graph": q["gold_in_graph"],
           "cascade_none": float(not p)}
    for m, ids in (("cascade", p), ("named", nn)):
        row[m], row[f"{m}_prec"], row[f"{m}_rec"] = link_eval(q, ids)
    LK["2wiki"].append(row)

def acc_line(label, rows, k):
    v = [r[k] for r in rows]; lo, hi = wilson(sum(v), len(v))
    return f"{label:<32}{len(v):>5}{np.mean(v):>9.1%} [{lo:.0%}-{hi:.0%}]"

LINKSUM = {}
print(f"{'MetaQA subset':<32}{'n':>5}{'cascade':>19}{'LLM-named':>19}{'cascade - named [95% CI]':>30}")
for lab, f in [("all", lambda r: True)] + [(f"{h}-hop", lambda r, h=h: r["hop"] == h) for h in (1, 2, 3)] + \
              [(f"topic: {t}", lambda r, t=t: r["ttype"] == t) for t in ("film", "person", "tag/other")]:
    rows = [r for r in LK["metaqa"] if f(r)]
    if not rows:
        continue
    d = paired_boot([r["named"] for r in rows], [r["cascade"] for r in rows])
    LINKSUM[("metaqa", lab)] = (len(rows), np.mean([r["cascade"] for r in rows]), np.mean([r["named"] for r in rows]), d)
    print(acc_line(lab, rows, "cascade") + acc_line("", rows, "named")[32 + 5:] + f"{f'{d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]':>30}")
print()
print(f"{'2Wiki subset (gold in graph)':<32}{'n':>5}{'cascade':>19}{'LLM-named':>19}{'cascade - named [95% CI]':>30}"
      f"{'P/R cascade':>14}{'P/R named':>12}")
for lab in ["all"] + sorted({q["group"] for q in W["questions"]}):
    rows = [r for r in LK["2wiki"] if r["gold_in_graph"] and lab in ("all", r["group"])]
    if not rows:
        continue
    d = paired_boot([r["named"] for r in rows], [r["cascade"] for r in rows])
    LINKSUM[("2wiki", lab)] = (len(rows), np.mean([r["cascade"] for r in rows]), np.mean([r["named"] for r in rows]), d)
    pr = lambda m: f"{np.nanmean([r[m + '_prec'] for r in rows]):.0%}/{np.nanmean([r[m + '_rec'] for r in rows]):.0%}"
    print(acc_line(lab, rows, "cascade") + acc_line("", rows, "named")[32 + 5:] +
          f"{f'{d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]':>30}{pr('cascade'):>14}{pr('named'):>12}")

MetaQA subset                       n            cascade          LLM-named      cascade - named [95% CI]
all                               900    99.9% [99%-100%]    99.0% [98%-99%]          +0.9% [+0.2%, +1.7%]
1-hop                             300   100.0% [99%-100%]   100.0% [99%-100%]          +0.0% [+0.0%, +0.0%]
2-hop                             300    99.7% [98%-100%]    99.0% [97%-100%]          +0.7% [-0.7%, +2.0%]
3-hop                             300   100.0% [99%-100%]    98.0% [96%-99%]          +2.0% [+0.7%, +3.7%]
topic: film                       566   100.0% [99%-100%]    98.9% [98%-100%]          +1.1% [+0.4%, +1.9%]


topic: person                     318    99.7% [98%-100%]    99.1% [97%-100%]          +0.6% [-0.6%, +1.9%]
topic: tag/other                   16   100.0% [81%-100%]   100.0% [81%-100%]          +0.0% [+0.0%, +0.0%]

2Wiki subset (gold in graph)        n            cascade          LLM-named      cascade - named [95% CI]   P/R cascade   P/R named
all                               956    88.7% [87%-91%]    77.8% [75%-80%]        +10.9% [+8.6%, +13.2%]       97%/94%     98%/85%
bridge_comparison                 234    80.8% [75%-85%]    72.6% [67%-78%]         +8.1% [+4.3%, +12.4%]      100%/93%    100%/87%
comparison                        223    98.2% [95%-99%]    88.8% [84%-92%]         +9.4% [+5.4%, +13.9%]       99%/99%     99%/94%
compositional                     396    87.9% [84%-91%]    75.3% [71%-79%]        +12.6% [+8.8%, +16.7%]       94%/91%     96%/80%
inference                         103    89.3% [82%-94%]    75.7% [67%-83%]        +13.6% [+6.8%, +21.4%]       94%/95%    

### Where the cascade decides, and how it fails

Per stage: the share of questions that stage decided and its accuracy there. Failure types: *no link* (the cascade
returned nothing) and *wrong link*. For MetaQA, *gold among candidates* says whether the LLM step could have picked the
gold entity at all.

In [11]:
for b in ("metaqa", "2wiki"):
    rows = [r for r in LK[b] if b == "metaqa" or r["gold_in_graph"]]
    print(f"{b}  (n = {len(rows)})")
    for st in ("exact", "alias", "embedding", "llm"):
        R = [r for r in rows if r["stage"] == st]
        if R:
            print(f"   {st:<10} decides {len(R) / len(rows):>6.1%} of questions, accuracy there {np.mean([r['cascade'] for r in R]):.1%}")
    wrong = [r for r in rows if not r["cascade"]]
    print(f"   failures: {len(wrong) / len(rows):.1%} of questions = no link {sum(r['cascade_none'] for r in wrong) / len(rows):.1%}"
          f" + wrong link {sum(1 - r['cascade_none'] for r in wrong) / len(rows):.1%}")
    if b == "metaqa":
        print(f"   gold entity among the cascade's candidates or link: {np.mean([r['gold_in_cands'] for r in rows]):.1%}; "
              f"LLM-named string is a graph key: {np.mean([r['named_in_graph'] for r in rows]):.1%}")
print("\nMetaQA cascade errors (first 8):")
for i, q in enumerate(QM):
    if not LK["metaqa"][i]["cascade"] and sum(not r["cascade"] for r in LK["metaqa"][:i]) < 8:
        p = MQ_NAMES[PRED["metaqa"][i][0]] if PRED["metaqa"][i] else None
        print(f"   [{LINK['metaqa'][i]['stage']}] {q['text']!r}: gold {q['topic']!r}, cascade {p!r}, LLM-named {NAMED['metaqa'][i]}")
print("2Wiki cascade errors (first 6):")
k = 0
for i, q in enumerate(W["questions"]):
    if q["gold_in_graph"] and not LK["2wiki"][i]["cascade"] and k < 6:
        k += 1
        print(f"   [{LINK['2wiki'][i]['stage']}] {q['question']!r}: gold {q['gold_names']}, "
              f"cascade {[W['ent_name'][j] for j in PRED['2wiki'][i]]}, LLM-named {NAMED['2wiki'][i]}")

metaqa  (n = 900)
   exact      decides  88.7% of questions, accuracy there 100.0%
   llm        decides  11.3% of questions, accuracy there 99.0%
   failures: 0.1% of questions = no link 0.1% + wrong link 0.0%
   gold entity among the cascade's candidates or link: 100.0%; LLM-named string is a graph key: 99.0%
2wiki  (n = 956)
   exact      decides  43.5% of questions, accuracy there 93.5%
   alias      decides   0.2% of questions, accuracy there 0.0%
   llm        decides  56.3% of questions, accuracy there 85.3%
   failures: 11.3% of questions = no link 0.4% + wrong link 10.9%

MetaQA cascade errors (first 8):
   [llm] 'what were the release years of the movies directed by Jude Law': gold 'Jude Law', cascade None, LLM-named ['Jude Law']
2Wiki cascade errors (first 6):
   [llm] "Which film has the director who was born later, Playing It Wild or I'Ll Be Going Now?": gold ['Playing It Wild', "I'll Be Going Now", 'William Duncan (actor)'], cascade ['Playing It Wild', "I'll Be Going Now"

## Downstream (MetaQA): Notebook 1.2b's graph agent with raw, gold, predicted and LLM-named grounding

Tool, prompt and step budget are Notebook 1.2b's, except that the system prompt says "starting from the entity the
question is about" instead of "the topic entity in [brackets]", and the user message is the bracket-free question,
followed by `Topic entity: <name>` in the grounded conditions. Every step is one `llm.run` over all active episodes
(one Batch job per step). Conditions whose user message is identical (for example a correct predicted link and the
gold link) share one episode, since the calls are deterministic.

In [12]:
MAX_LIST, MAX_STEPS = 150, 8
ANSWER_RULES = ("Return JSON {\"answers\": [...]} listing EVERY entity that answers the question, using the exact "
                "entity names; put the most likely answer first; do not include the topic entity itself unless it is "
                "an answer; return an empty list if you cannot find the answer.")

class GraphTool:
    def __init__(self, G):
        self.G, self.lookup = G, {}
        for n in G.nodes():
            self.lookup.setdefault(key(n), n)
    def __call__(self, entities, relation=None):
        if isinstance(entities, str):
            entities = [entities]
        out, missing = defaultdict(set), 0
        for nm in entities:
            e = nm if nm in self.G else self.lookup.get(key(nm))
            if e is None:
                missing += 1; continue
            for _, t, r in self.G.out_edges(e, keys=True):
                if relation in (None, r): out[f"{r} (outgoing: the movie's {r.replace('_', ' ')} value)"].add(t)
            for s, _, r in self.G.in_edges(e, keys=True):
                if relation in (None, r): out[f"{r} (incoming: movies with this {r.replace('_', ' ')} value)"].add(s)
        result = {k: {"count": len(v), "entities": sorted(v)[:MAX_LIST]} for k, v in out.items()}
        if missing:
            result["not_found"] = missing
        if not out:
            result["note"] = "no edges with this relation for these entities; try another relation or omit it"
        return json.dumps(result, ensure_ascii=False)

TOOL = GraphTool(KB)
TOOLS = [{"type": "function", "function": {
    "name": "get_neighbors",
    "description": ("Traverse the movie knowledge graph. Movies are linked to their directors, writers, actors, release "
                    "year, language, genres and tags (relations: directed_by, written_by, starred_actors, release_year, "
                    "in_language, has_genre, has_tags, has_imdb_rating, has_imdb_votes). Returns neighbours in both "
                    "directions, labelled outgoing (movie -> value) or incoming (value -> movies). Pass several "
                    "entities at once to expand a whole set."),
    "parameters": {"type": "object", "properties": {
        "entities": {"type": "array", "items": {"type": "string"}},
        "relation": {"type": "string", "enum": sorted(RELS)}},
        "required": ["entities"]}}}]
AGENT_SYS = ("You answer movie questions by traversing a knowledge graph with the get_neighbors tool, one hop at a "
             "time, starting from the entity the question is about. " + ANSWER_RULES)

def body_json(msgs, tools=None, max_tokens=1500):
    b = LLM.body(msgs, max_tokens=max_tokens, tools=tools)
    b["response_format"] = {"type": "json_object"}
    return b

M_COND = ("raw", "gold", "predicted", "llm_named")
HINT = {}
for i, q in enumerate(QM):
    HINT[("raw", q["qid"])] = None
    HINT[("gold", q["qid"])] = q["topic"]
    HINT[("predicted", q["qid"])] = MQ_NAMES[PRED["metaqa"][i][0]] if PRED["metaqa"][i] else None
    HINT[("llm_named", q["qid"])] = NAMED["metaqa"][i][0] if NAMED["metaqa"][i] else None
user_msg = lambda q, h: q["text"] + (f"\nTopic entity: {h}" if h else "")
EP = {(c, q["qid"]): f"{q['qid']}|{hashlib.sha1(user_msg(q, HINT[(c, q['qid'])]).encode()).hexdigest()[:10]}"
      for c in M_COND for q in QM}
QBY = {q["qid"]: q for q in QM}
state = {}
for (c, qid), ep in EP.items():
    state.setdefault(ep, {"msgs": [{"role": "system", "content": AGENT_SYS},
                                   {"role": "user", "content": user_msg(QBY[qid], HINT[(c, qid)])}],
                          "answer": None, "calls": 0, "tokens": 0})
print(f"{len(EP):,} (condition, question) pairs -> {len(state):,} distinct episodes")
for step in range(MAX_STEPS + 1):
    active = [k for k, s in state.items() if s["answer"] is None]
    if not active:
        break
    res = llm_m.run(f"nb21_metaqa_agent_step{step + 1}",
                    {k: body_json(state[k]["msgs"], tools=TOOLS if step < MAX_STEPS else None) for k in active})
    for k in active:
        r, s = res[k], state[k]
        s["tokens"] += r.get("in", 0) + r.get("out", 0)
        if not r.get("tool_calls"):
            s["answer"] = r.get("text") or "{}"; continue
        s["msgs"].append({"role": "assistant", "content": r["text"] or None,
                          "tool_calls": [{"id": t["id"], "type": "function", "function": {"name": t["name"], "arguments": t["arguments"]}}
                                         for t in r["tool_calls"]]})
        for t in r["tool_calls"]:
            s["calls"] += 1
            try:
                out = TOOL(**json.loads(t["arguments"]))
            except Exception as e:
                out = json.dumps({"error": f"{type(e).__name__}: {e}"})
            s["msgs"].append({"role": "tool", "tool_call_id": t["id"], "content": out})
    print(f"step {step + 1}: {len(active)} episodes active")

3,600 (condition, question) pairs -> 1,817 distinct episodes


step 1: 1817 episodes active


step 2: 1817 episodes active


step 3: 1238 episodes active


step 4: 630 episodes active
step 5: 121 episodes active
step 6: 64 episodes active
step 7: 42 episodes active


step 8: 31 episodes active
step 9: 24 episodes active


In [13]:
def parse_answers(text):
    try:
        return [str(a) for a in json.loads(text).get("answers", [])]
    except Exception:
        return []

def score_m(pred, gold):
    P, Gs = [key(p) for p in pred], {key(g) for g in gold}
    tp = len(set(P) & Gs)
    prec, rec = (tp / len(set(P)) if P else 0.0), tp / len(Gs)
    return {"hits1": float(bool(P) and P[0] in Gs), "f1": 2 * prec * rec / (prec + rec) if tp else 0.0}

SM = []
for i, q in enumerate(QM):
    row = {"hop": q["hop"], "ttype": q["ttype"], "link_ok": LK["metaqa"][i]["cascade"]}
    for c in M_COND:
        st = state[EP[(c, q["qid"])]]
        row[c] = {**score_m(parse_answers(st["answer"]), q["gold"]), "calls": st["calls"], "tokens": st["tokens"]}
    SM.append(row)

COMP_M = [("predicted", "raw"), ("gold", "predicted"), ("llm_named", "raw"), ("predicted", "llm_named")]
RES_M = {}
print(f"{'MetaQA':<10}{'n':>5}" + "".join(f"{c + ' Hits@1':>18}" for c in M_COND) +
      "".join(f"{a + ' - ' + b:>28}" for a, b in COMP_M))
for lab, f in [(f"{h}-hop", lambda r, h=h: r["hop"] == h) for h in (1, 2, 3)] + [("all", lambda r: True)]:
    rows = [r for r in SM if f(r)]
    means = {c: np.mean([r[c]["hits1"] for r in rows]) for c in M_COND}
    diffs = {(a, b): paired_boot([r[b]["hits1"] for r in rows], [r[a]["hits1"] for r in rows]) for a, b in COMP_M}
    RES_M[lab] = (len(rows), means, diffs)
    print(f"{lab:<10}{len(rows):>5}" + "".join(f"{means[c]:>18.1%}" for c in M_COND) +
          "".join(f"{f'{d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]':>28}" for d in diffs.values()))
print("\nF1 (all):", ", ".join(f"{c} {np.mean([r[c]['f1'] for r in SM]):.3f}" for c in M_COND))
print("tool calls / tokens per question:", ", ".join(
    f"{c} {np.mean([r[c]['calls'] for r in SM]):.1f} / {np.mean([r[c]['tokens'] for r in SM]):,.0f}" for c in M_COND))
for ok in (1.0, 0.0):
    rows = [r for r in SM if r["link_ok"] == ok]
    if rows:
        print(f"questions where the cascade link is {'correct' if ok else 'wrong or missing'} (n = {len(rows)}): " +
              ", ".join(f"{c} {np.mean([r[c]['hits1'] for r in rows]):.1%}" for c in M_COND))

MetaQA        n        raw Hits@1       gold Hits@1  predicted Hits@1  llm_named Hits@1             predicted - raw            gold - predicted             llm_named - raw       predicted - llm_named
1-hop       300             91.0%             90.3%             90.0%             90.3%        -1.0% [-2.7%, +0.7%]        +0.3% [+0.0%, +1.0%]        -0.7% [-2.7%, +1.0%]        -0.3% [-1.0%, +0.0%]


2-hop       300             97.0%             97.0%             97.0%             97.0%        +0.0% [-1.3%, +1.3%]        +0.0% [+0.0%, +0.0%]        +0.0% [-1.3%, +1.3%]        +0.0% [-1.0%, +1.0%]
3-hop       300             93.3%             94.3%             94.0%             94.0%        +0.7% [-2.3%, +3.7%]        +0.3% [+0.0%, +1.0%]        +0.7% [-2.3%, +3.7%]        +0.0% [-1.0%, +1.0%]


all         900             93.8%             93.9%             93.7%             93.8%        -0.1% [-1.3%, +1.1%]        +0.2% [+0.0%, +0.6%]        +0.0% [-1.2%, +1.2%]        -0.1% [-0.7%, +0.3%]

F1 (all): raw 0.946, gold 0.950, predicted 0.947, llm_named 0.948
tool calls / tokens per question: raw 2.3 / 1,892, gold 2.3 / 1,914, predicted 2.3 / 1,906, llm_named 2.3 / 1,927
questions where the cascade link is correct (n = 899): raw 93.8%, gold 93.9%, predicted 93.7%, llm_named 93.8%
questions where the cascade link is wrong or missing (n = 1): raw 100.0%, gold 100.0%, predicted 100.0%, llm_named 100.0%


## Downstream (2Wiki): Notebook 1.4a's retriever and reader, seeded from raw text or from links

`retrieve` is Notebook 1.4a's (dense top 5, and PPR seeded from the five triples most similar to the question); its
answers are read from Notebook 1.4a's cache (identical requests). `retrieve_linked` runs the same PPR with the linked
entity nodes as seeds (weight 1 each) plus the same dense passage prior; with no link, only the prior remains.

In [14]:
K = 5

def ppr(T, pers, alpha=0.5, iters=40):
    p = pers / pers.sum(); x = p.copy()
    for _ in range(iters):
        x = (1 - alpha) * p + alpha * (T @ x)
    return x

def retrieve(d, qi):
    q = d["Q"][qi]
    dense = d["P"] @ q
    vec = list(np.argsort(-dense)[:K])
    pers = np.zeros(d["T"].shape[0], np.float64)
    fs = d["F"] @ q if len(d["facts"]) else np.zeros(0)
    for j in np.argsort(-fs)[:5]:
        a, b, pi, _ = d["facts"][j]
        pers[a] += fs[j]; pers[b] += fs[j]
    for j in np.argsort(-dense)[:20]:
        pers[j] += 0.05 * max(dense[j], 0)
    if pers.sum() == 0:
        return vec, vec
    x = ppr(d["T"], pers)[:d["n_p"]]
    return vec, list(np.argsort(-x)[:K])

def retrieve_linked(d, qi, ents):
    dense = d["P"] @ d["Q"][qi]
    pers = np.zeros(d["T"].shape[0], np.float64)
    for j in ents:
        pers[d["n_p"] + j] += 1.0
    for j in np.argsort(-dense)[:20]:
        pers[j] += 0.05 * max(dense[j], 0)
    x = ppr(d["T"], pers)[:d["n_p"]]
    return [int(i) for i in np.argsort(-x)[:K]]

def recall(d, q, ids):
    if not q["support"]:
        return float("nan")
    got = {d["passages"][i] for i in ids}
    return len(q["support"] & got) / len(q["support"])

W_COND = ("vector", "graph_raw", "gold", "predicted", "llm_named")
RET = {}
for qi, q in enumerate(W["questions"]):
    v, g = retrieve(W, qi)
    RET[("vector", qi)], RET[("graph_raw", qi)] = [int(i) for i in v], [int(i) for i in g]
    RET[("gold", qi)] = retrieve_linked(W, qi, [j for j in q["gold_ents"] if j is not None])
    RET[("predicted", qi)] = retrieve_linked(W, qi, PRED["2wiki"][qi])
    RET[("llm_named", qi)] = retrieve_linked(W, qi, named_to_ids("2wiki", NAMED["2wiki"][qi]))
for c in W_COND:
    print(f"recall@5 of gold supporting paragraphs, {c:<10} {np.nanmean([recall(W, q, RET[(c, qi)]) for qi, q in enumerate(W['questions'])]):.1%}")

recall@5 of gold supporting paragraphs, vector     70.6%
recall@5 of gold supporting paragraphs, graph_raw  84.4%
recall@5 of gold supporting paragraphs, gold       78.1%
recall@5 of gold supporting paragraphs, predicted  76.3%
recall@5 of gold supporting paragraphs, llm_named  75.6%


In [15]:
ANSWER_SYS = ("Answer the question using the passages. Reason step by step in at most three short sentences, then "
              "give the final answer on the last line as 'Answer: <answer>'. The answer must be a short phrase "
              "(an entity, date, number or yes/no), not a sentence.")

def ctx(d, ids):
    return "\n\n".join(f"[{n + 1}] {d['passages'][i]}" for n, i in enumerate(ids))

def final(text):
    m = re.findall(r"Answer:\s*(.+)", text)
    return (m[-1] if m else text.strip().splitlines()[-1] if text.strip() else "").strip()

def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = Counter(p) & Counter(g); same = sum(common.values())
    if same == 0:
        return 0.0
    pr, rc = same / len(p), same / len(g)
    return 2 * pr * rc / (pr + rc)

ans_body = lambda qi, ids: LLM.body([{"role": "system", "content": ANSWER_SYS},
                                     {"role": "user", "content": f"{ctx(W, ids)}\n\nQuestion: {W['questions'][qi]['question']}"}],
                                    max_tokens=300)
# Raw-text conditions: Notebook 1.4a's stage, identical requests (read from its cache).
old = llm_w.run(f"{NAME}_answers", {f"{m}|{q['qid']}": ans_body(qi, RET[(c, qi)])
                                    for qi, q in enumerate(W["questions"]) for m, c in (("vector", "vector"), ("graph", "graph_raw"))})
# Linked conditions: one request per distinct (question, retrieved passages); identical retrievals share it.
AKEY = {(c, qi): f"{q['qid']}|{','.join(map(str, RET[(c, qi)]))}" for c in ("gold", "predicted", "llm_named")
        for qi, q in enumerate(W["questions"])}
new = llm_w.run("nb21_2wiki_answers", {k: ans_body(qi, RET[(c, qi)]) for (c, qi), k in AKEY.items()})
print(f"{len(AKEY):,} linked (condition, question) pairs -> {len(set(AKEY.values())):,} distinct answer requests")

SW = []
for qi, q in enumerate(W["questions"]):
    row = {"group": q["group"], "link_ok": LK["2wiki"][qi]["cascade"], "gold_in_graph": q["gold_in_graph"]}
    for c in W_COND:
        r = old[f"{'vector' if c == 'vector' else 'graph'}|{q['qid']}"] if c in ("vector", "graph_raw") else new[AKEY[(c, qi)]]
        pred = final(r["text"])
        row[c] = {"f1": max(f1(pred, g) for g in q["golds"]), "em": float(max(normalize(pred) == normalize(g) for g in q["golds"])),
                  "recall": recall(W, q, RET[(c, qi)])}
    SW.append(row)

COMP_W = [("predicted", "graph_raw"), ("predicted", "vector"), ("gold", "predicted"), ("llm_named", "graph_raw"),
          ("predicted", "llm_named")]
RES_W = {}
print(f"\n{'2Wiki':<18}{'n':>5}" + "".join(f"{c + ' F1':>14}" for c in W_COND) + "".join(f"{a + ' - ' + b:>28}" for a, b in COMP_W))
for lab in sorted({q["group"] for q in W["questions"]}) + ["all"]:
    rows = [r for r in SW if lab in ("all", r["group"])]
    means = {c: np.mean([r[c]["f1"] for r in rows]) for c in W_COND}
    diffs = {(a, b): paired_boot([r[b]["f1"] for r in rows], [r[a]["f1"] for r in rows]) for a, b in COMP_W}
    RES_W[lab] = (len(rows), means, diffs)
    print(f"{lab:<18}{len(rows):>5}" + "".join(f"{means[c]:>14.1%}" for c in W_COND) +
          "".join(f"{f'{d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]':>28}" for d in diffs.values()))
print("\nEM (all):", ", ".join(f"{c} {np.mean([r[c]['em'] for r in SW]):.1%}" for c in W_COND))

[nb21_2wiki_answers] 1,272 new requests -> batch (0 cached)


[nb21_2wiki_answers] resuming: 1 running, 0 queued


[nb21_2wiki_answers] batch batch_6aeadfb73ae08190a1552786182cf77c cancelled: 1,271 ok, 0 failed


[nb21_2wiki_answers] 1 min: 0 running, 0 queued, 1,272 results cached


[nb21_2wiki_answers] re-submitting 1 failed request(s)


[nb21_2wiki_answers] uploaded input file 1/1 (1 requests, 0 MB)


... (9 outputs left out)

[nb21_2wiki_answers] done in 27.3 min


3,000 linked (condition, question) pairs -> 1,272 distinct answer requests



2Wiki                 n     vector F1  graph_raw F1       gold F1  predicted F1  llm_named F1       predicted - graph_raw          predicted - vector            gold - predicted       llm_named - graph_raw       predicted - llm_named
bridge_comparison   235         61.9%         67.6%         69.7%         69.4%         68.5%        +1.8% [-5.9%, +9.6%]       +7.4% [+0.3%, +14.6%]        +0.4% [-2.8%, +3.4%]        +0.9% [-6.8%, +8.6%]        +0.9% [-1.7%, +3.5%]
comparison          244         89.2%         84.2%         92.8%         93.5%         92.7%       +9.3% [+4.8%, +13.9%]        +4.3% [+0.3%, +8.4%]        -0.7% [-2.0%, +0.2%]       +8.5% [+3.8%, +13.3%]        +0.8% [-1.2%, +2.9%]
compositional       413         23.8%         47.6%         34.0%         32.3%         31.3%     -15.4% [-19.8%, -11.0%]       +8.5% [+4.8%, +12.2%]        +1.8% [-0.1%, +3.8%]     -16.3% [-20.8%, -11.9%]        +1.0% [-0.7%, +2.7%]
inference           108         47.2%         62.9%         55.

all                1000         51.2%         62.9%         59.0%         58.5%         57.5%        -4.4% [-7.5%, -1.4%]        +7.3% [+4.7%, +9.9%]        +0.5% [-0.6%, +1.7%]        -5.4% [-8.5%, -2.4%]        +1.1% [-0.0%, +2.1%]

EM (all): vector 45.0%, graph_raw 53.3%, gold 50.9%, predicted 50.9%, llm_named 49.7%


## What this test cost

Only this notebook's stages are paid here (`nb21_*`); the 2Wiki passages, extraction, graph and raw-text answers
are reused from Notebook 1.4a. Embedding cost counts only this notebook's embeddings (MetaQA entity names, question
n-grams).

In [16]:
def stage_names(llm):
    return sorted(f[:-6] for f in os.listdir(llm.cache) if f.startswith("nb21_") and f.endswith(".jsonl"))

def emb_usd(llm):
    tok = sum(json.load(open(f"{llm.cache}/{f}"))["tokens"] for f in os.listdir(llm.cache)
              if f.startswith("nb21_") and f.endswith(".embed.json"))
    return tok, tok * EMBED_PRICE / 1e6

COST = {}
print(f"{'stage':<36}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}{'via Batch':>11}")
for b, llm in (("metaqa", llm_m), ("2wiki", llm_w)):
    led = llm.ledger(stage_names(llm))
    led.pop("(embeddings)", None)
    for s, a in led.items():
        print(f"{s:<36}{a['calls']:>8,}{a['in']:>13,}{a['out']:>12,}{a['usd']:>8.3f}{a['batch_share']:>11.0%}")
    et, eu = emb_usd(llm)
    print(f"{'nb21 embeddings (' + b + ')':<36}{'':>8}{et:>13,}{'':>12}{eu:>8.3f}")
    COST[b] = sum(a["usd"] for a in led.values()) + eu
n_items = {"metaqa": len(QM), "2wiki": len(W["questions"])}
full = {"metaqa": 3 * N_PER_HOP, "2wiki": 1000}
print(f"{'TOTAL (new calls only)':<36}{'':>33}{sum(COST.values()):>8.3f}")
print("Cost per question: " + ", ".join(f"{b} ${COST[b] / n_items[b]:.5f}" for b in COST))
if TEST:
    # Test calls are standard tier (2x Batch); the full run goes through the Batch API.
    fc = {b: COST[b] / n_items[b] * full[b] / 2 for b in COST}
    print("Forecast of the full Batch run (linear in n, Batch prices): " +
          ", ".join(f"{b} ${v:.2f}" for b, v in fc.items()) + f", total ${sum(fc.values()):.2f}")

stage                                  calls    input tok  output tok     USD  via Batch
nb21_metaqa_agent_step1                1,817      664,952      47,619   0.045       100%
nb21_metaqa_agent_step2                1,817      808,422      56,420   0.055       100%
nb21_metaqa_agent_step3                1,238      729,143      87,276   0.058       100%
nb21_metaqa_agent_step4                  630      524,522      39,793   0.036       100%
nb21_metaqa_agent_step5                  121      155,828      13,563   0.011       100%
nb21_metaqa_agent_step6                   64      106,501       4,871   0.007       100%
nb21_metaqa_agent_step7                   42       86,698       3,680   0.005       100%
nb21_metaqa_agent_step8                   31       71,655       2,424   0.004       100%
nb21_metaqa_agent_step9                   24       61,750       1,954   0.004       100%
nb21_metaqa_disambiguate                 102       44,839       1,121   0.003       100%
nb21_metaqa_llm_named

## Limitations

- **Gold links.** MetaQA's brackets give the exact KB name, so its gold link is exact; the template filter uses
  question statistics (no labels) and MetaQA's templates make it unusually effective. 2Wiki's gold links are the
  evidence subjects mapped to an LLM-extracted graph by normalised name; a question whose gold entity the extraction
  missed or spelled differently is left out of linking accuracy (its share is printed) but kept downstream.
- **Linked retrieval on 2Wiki is one design.** Entity seeds replace Notebook 1.4a's triple seeds; HippoRAG 2 found
  query→triple seeding better than NER→node seeding, so *graph-raw* is a strong raw-text baseline. Combining both
  seed types was not tested.
- **Raw-text MetaQA agent.** It can find the entity itself by calling the tool with names from the question, so *raw*
  measures what grounding adds on top of an agent that grounds implicitly, not over an ungrounded system.
- **Fixed thresholds.** 1 % template frequency, cosine 0.9 (Notebook 1.4a's synonym threshold), 10 embedding
  candidates; fixed before the run, not tuned. In the test run the 2Wiki graph is built from the 25 test questions'
  passages only, so linking there is easier than in the full pool.
- **One model, one run.** `gpt-6-luna`, temperature 0; one prompt per stage.

**References.** Gutiérrez et al., *HippoRAG* (NeurIPS 2024) and *From RAG to Memory* (HippoRAG 2, ICML 2025) ·
Xiong et al., *Interactive-KBQA* (ACL 2024) · Luo et al., *ChatKBQA* (Findings ACL 2024) · Luo et al., *An Entity
Linking Agent for Question Answering* (2025) · Xu et al., *Fine-tuned LLMs Know More, Hallucinate Less* (WikiSP).

In [17]:
print("=" * 120)
print(f"RESULTS SUMMARY — Notebook 1.6: grounding questions in graph entities, {MODE} run, gpt-6-luna (no reasoning)")
print("=" * 120)
fmt = lambda d: f"{d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]"
for b, lab in (("metaqa", "all"), ("2wiki", "all")):
    n, c, m, d = LINKSUM[(b, lab)]
    print(f"Linking accuracy {b:<7} n={n:<5} cascade {c:.1%} | LLM-named {m:.1%} | cascade - named {fmt(d)}")
n, means, diffs = RES_M["all"]
print(f"MetaQA Hits@1    n={n:<5} " + " | ".join(f"{c} {means[c]:.1%}" for c in M_COND))
for (a, b_), d in diffs.items():
    print(f"   {a} - {b_}: {fmt(d)}")
n, means, diffs = RES_W["all"]
print(f"2Wiki F1         n={n:<5} " + " | ".join(f"{c} {means[c]:.1%}" for c in W_COND))
for (a, b_), d in diffs.items():
    print(f"   {a} - {b_}: {fmt(d)}")
prim = {"MetaQA (Hits@1, predicted - raw)": RES_M["all"][2][("predicted", "raw")],
        "2Wiki (F1, predicted - graph_raw)": RES_W["all"][2][("predicted", "graph_raw")]}
print("\nPrimary verdict: " + "; ".join(
    f"{k}: {'better' if t[1] > 0 else 'worse' if t[2] < 0 else 'no resolvable difference'}" for k, t in prim.items()))
print(f"Cost of new calls in this run: ${sum(COST.values()):.3f}")
print("\nLecture message (Task 1.6): a graph method is only as good as its entry point. The linking-accuracy line says how"
      "\noften a cheap cascade finds that entry point; the gold - predicted lines say what its misses cost downstream.")

RESULTS SUMMARY — Notebook 1.6: grounding questions in graph entities, batch run, gpt-6-luna (no reasoning)
Linking accuracy metaqa  n=900   cascade 99.9% | LLM-named 99.0% | cascade - named +0.9% [+0.2%, +1.7%]
Linking accuracy 2wiki   n=956   cascade 88.7% | LLM-named 77.8% | cascade - named +10.9% [+8.6%, +13.2%]
MetaQA Hits@1    n=900   raw 93.8% | gold 93.9% | predicted 93.7% | llm_named 93.8%
   predicted - raw: -0.1% [-1.3%, +1.1%]
   gold - predicted: +0.2% [+0.0%, +0.6%]
   llm_named - raw: +0.0% [-1.2%, +1.2%]
   predicted - llm_named: -0.1% [-0.7%, +0.3%]
2Wiki F1         n=1000  vector 51.2% | graph_raw 62.9% | gold 59.0% | predicted 58.5% | llm_named 57.5%
   predicted - graph_raw: -4.4% [-7.5%, -1.4%]
   predicted - vector: +7.3% [+4.7%, +9.9%]
   gold - predicted: +0.5% [-0.6%, +1.7%]
   llm_named - graph_raw: -5.4% [-8.5%, -2.4%]
   predicted - llm_named: +1.1% [-0.0%, +2.1%]

Primary verdict: MetaQA (Hits@1, predicted - raw): no resolvable difference; 2Wiki (F1, predic